<a href="https://colab.research.google.com/github/WhySoSerious404/trust-safety-analytics/blob/main/02-abuse-detection-sql/abuse_detection_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Abuse detection in SQL

Generate synthetic platform activity, detect planted patterns
using SQL, and evaluate account-level flags.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR = Path(
    "/content/drive/MyDrive/Project: trust-safety/"
    "02-abuse-detection-sql/data"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data folder:', DATA_DIR)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/Project: trust-safety/02-abuse-detection-sql/data


In [2]:
## Synthetic data generator

In [3]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

N_ACCOUNTS = 20000
TARGET_EVENTS = 300000

START = pd.Timestamp("2026-09-01", tz="UTC")
END = START + pd.Timedelta(days=30)

PATTERN_COUNTS = {
    "legitimate": 19000,
    "spam": 400,
    "takeover": 300,
    "automated_clicking": 300
}

assert sum(PATTERN_COUNTS.values()) == N_ACCOUNTS

print("Accounts:", N_ACCOUNTS)
print("Target events:", TARGET_EVENTS)
print("Observation period:", START, "to", END, "(end excluded)")

pd.Series(PATTERN_COUNTS, name="account_count")

Accounts: 20000
Target events: 300000
Observation period: 2026-09-01 00:00:00+00:00 to 2026-10-01 00:00:00+00:00 (end excluded)


,account_count
legitimate,19000
spam,400
takeover,300
automated_clicking,300


In [4]:
account_ids = np.arange(1, N_ACCOUNTS + 1)

signup_dates = START - pd.to_timedelta(
    rng.integers(1, 61, size=N_ACCOUNTS),
    unit="D"
)

accounts = pd.DataFrame({
    "account_id": account_ids,
    "signup_date": signup_dates.strftime("%Y-%m-%d"),
    "country": rng.choice(
        ["IN", "US", "GB", "DE", "CA"],
        size=N_ACCOUNTS
    )
})

labels = np.repeat(
    list(PATTERN_COUNTS.keys()),
    list(PATTERN_COUNTS.values())
)

rng.shuffle(labels)

ground_truth = pd.DataFrame({
    "account_id": account_ids,
    "pattern": labels
})

assert accounts["account_id"].is_unique
assert len(accounts) == N_ACCOUNTS

display(accounts.head())
ground_truth["pattern"].value_counts()

,account_id,signup_date,country
0,1,2026-08-26,IN
1,2,2026-07-16,DE
2,3,2026-07-23,US
3,4,2026-08-05,DE
4,5,2026-08-06,GB


,count
pattern,
legitimate,19000
spam,400
takeover,300
automated_clicking,300
